# Data Access Guide

This notebook shows you how to programmatically download several of the
materials science datasets recommended for the course project, and how to
inspect and prepare them for analysis.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='ticks', palette='colorblind')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})


## A. MatBench / matminer datasets (recommended for most projects)

In [ ]:
# pip install matminer   (already installed on the course server)
from matminer.datasets import load_dataset, get_available_datasets

# List all available datasets
available = sorted(get_available_datasets())
print(f"{len(available)} datasets available in matminer. First 20:")
for name in available[:20]:
    print(f"  {name}")
print("  ...")


In [ ]:
# ── Load the steel strength dataset ──────────────────────────────────────────
df_steel = load_dataset("steel_strength")
print(f"Shape: {df_steel.shape}")
print(f"Columns: {df_steel.columns.tolist()}")
print()
print(df_steel.describe().round(2))


In [ ]:
# Quick pairplot to visualise all pairwise relationships
feature_cols = [c for c in df_steel.columns if c not in ['formula']]
g = sns.pairplot(df_steel[feature_cols].dropna(), diag_kind='kde',
                  plot_kws={'alpha': 0.4, 's': 20},
                  diag_kws={'linewidth': 1.5})
g.fig.suptitle('Steel Strength Dataset — Pairplot', y=1.02)
plt.show()
print(f"Pairplot shows {df_steel.shape[1]-1} numeric variables.")


## B. UCI Machine Learning Repository

In [ ]:
# pip install ucimlrepo   (already installed on the course server)
from ucimlrepo import fetch_ucirepo

# ── Concrete compressive strength (UCI ID 165) ────────────────────────────────
concrete = fetch_ucirepo(id=165)
df_concrete = concrete.data.features.copy()
df_concrete['strength_MPa'] = concrete.data.targets.values

print("Concrete dataset:")
print(df_concrete.head())
print()
print(df_concrete.describe().round(2))


In [ ]:
# ── Glass identification (UCI ID 42) ─────────────────────────────────────────
glass = fetch_ucirepo(id=42)
df_glass = glass.data.features.copy()
df_glass['Type'] = glass.data.targets.values.ravel()

type_names = {1: 'Building float', 2: 'Building non-float',
              3: 'Vehicle float', 5: 'Containers', 6: 'Tableware', 7: 'Headlamps'}
df_glass['Type_name'] = df_glass['Type'].map(type_names)

print("Glass dataset:")
print(df_glass.head())
print(f"\nClass counts:\n{df_glass['Type_name'].value_counts()}")


In [ ]:
# ── Quick bar chart: glass types ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
df_glass['Type_name'].value_counts().plot(kind='bar', ax=ax, color=sns.color_palette('colorblind'))
ax.set_xlabel('Glass type')
ax.set_ylabel('Count')
ax.set_title('Class distribution — Glass dataset')
ax.tick_params(axis='x', rotation=30)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()


## C. Materials Project (requires a free API key)

In [ ]:
# 1. Register at https://materialsproject.org and copy your API key
# 2. pip install mp-api

# ─────────────────────────────────────────────────────────────────────────────
# IMPORTANT: replace the string below with your own key before running
# ─────────────────────────────────────────────────────────────────────────────
MP_API_KEY = "YOUR_API_KEY_HERE"

if MP_API_KEY != "YOUR_API_KEY_HERE":
    try:
        from mp_api.client import MPRester
        with MPRester(MP_API_KEY) as mpr:
            docs = mpr.summary.search(
                elements=["Fe", "O"],
                fields=["material_id", "formula_pretty",
                        "band_gap", "formation_energy_per_atom",
                        "energy_above_hull", "density"]
            )
        df_mp = pd.DataFrame([{
            'id':        d.material_id,
            'formula':   d.formula_pretty,
            'band_gap':  d.band_gap,
            'Ef_eV_at':  d.formation_energy_per_atom,
            'Ehull_eV':  d.energy_above_hull,
            'density':   d.density,
        } for d in docs])
        print(f"Downloaded {len(df_mp)} Fe–O compounds")
        print(df_mp.head())
    except Exception as e:
        print(f"Error: {e}")
else:
    print("Set MP_API_KEY to your key from materialsproject.org to run this cell.")


## D. Zenodo — downloading a CSV from a published dataset

In [ ]:
# Most Zenodo datasets are plain CSV/Excel files accessible via a direct URL.
# Example (replace with the actual DOI-linked URL from the paper):

import urllib.request, io

# Placeholder URL — replace with the real dataset URL from the paper's README
sample_url = "https://raw.githubusercontent.com/scikit-learn/scikit-learn/main/sklearn/datasets/data/iris.csv"

try:
    with urllib.request.urlopen(sample_url) as response:
        raw = response.read().decode('utf-8')
    df_zenodo = pd.read_csv(io.StringIO(raw), header=None)
    print(f"Downloaded {len(df_zenodo)} rows")
    print(df_zenodo.head())
except Exception as e:
    print(f"Could not download: {e}")


## Tips for Your Own Dataset

1. **Always check units** — mix of SI and non-SI units in the same column is a
   common source of erroneous regression coefficients.
2. **Handle missing values** — use `df.isna().sum()` to count NaNs per column;
   either drop rows or impute with the column mean/median.
3. **Save a clean copy** — after preprocessing, save with
   `df_clean.to_csv("data_clean.csv", index=False)` so the notebook can be
   rerun from this point without hitting the network.
4. **Cite your source** — note the dataset DOI or URL in a markdown cell near
   the top of your project notebook.
5. **Write down the metadata, not just the data** — source, licence, and a
   one-line definition of every column, the same way Part II's `fair_check`
   and `validate_entry` functions do. A marker (and future you) should be
   able to answer "where did this come from and what does column `Eg` mean?"
   without re-reading the whole notebook.

```python
# Minimal preprocessing template
df_clean = (df.dropna()                        # remove rows with any NaN
              .reset_index(drop=True))         # reset integer index
print(f"After cleaning: {df_clean.shape}")
print(df_clean.dtypes)
```